In [1]:
import pandas as pd
import numpy as np
import string
import spacy

In [2]:
import gensim
from gensim.models import Word2Vec
from gensim import corpora
from gensim.utils import simple_preprocess
from gensim import models
from gensim.corpora import Dictionary
from gensim.parsing.preprocessing import STOPWORDS
from gensim.models import TfidfModel
from gensim import matutils

In [3]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfTransformer , CountVectorizer

In [24]:
from catboost import CatBoostClassifier

In [4]:
dataset = pd.read_csv('/home/pavel/Загрузки/Telegram Desktop/IMDB Dataset.csv')

In [5]:
dataset

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [6]:
dataset['sentiment'] = dataset['sentiment'].map({'positive' : 1, 'negative' : 0 })

In [7]:
df = dataset.copy()

In [8]:
def cleaner_text(texts) -> list:
    clean_text = []
    for text in texts:
        punkt = ''.join(cln for cln in text if cln not in string.punctuation)
        number = ''.join(num for num in punkt if num not in string.digits)
        char =  number.replace('br', '')
        char = char.lower()
        clean_text.append(char)
    return clean_text

clean_text = cleaner_text(df['review'])

In [9]:
def lemmatized_and_tokens_text(texts:list) -> list:
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens_text = []
    
    for text in texts:
        tokens_clean_text = word_tokenize(text)
        filterd = [word for word in tokens_clean_text if word not in stop_words]
        lemmatized = [lemmatizer.lemmatize(word) for word in filterd]
        lemmatized_tokens_text.append(lemmatized)

    return lemmatized_tokens_text

token_lemmatized_text = lemmatized_and_tokens_text(clean_text)

In [10]:
clean_dataset = pd.DataFrame({
    'review' :  [loc for loc in dataset['review']],
    'clean_review' : [loc for loc in clean_text],
    'lemmatized_tokens' : [loc for loc in token_lemmatized_text],
    'sentiment' : [sent for sent in dataset['sentiment']]
                     })

In [11]:
clean_dataset

,review,clean_review,lemmatized_tokens,sentiment
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,"[one, reviewer, mentioned, watching, oz, episo...",1
1,A wonderful little production. <br /><br />The...,a wonderful little production the filming te...,"[wonderful, little, production, filming, techn...",1
2,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,"[thought, wonderful, way, spend, time, hot, su...",1
3,Basically there's a family where a little boy ...,basically theres a family where a little boy j...,"[basically, there, family, little, boy, jake, ...",0
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love in the time of money is a ...,"[petter, matteis, love, time, money, visually,...",1
...,...,...,...,...
49995,I thought this movie did a down right good job...,i thought this movie did a down right good job...,"[thought, movie, right, good, job, wasnt, crea...",1
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",bad plot bad dialogue bad acting idiotic direc...,"[bad, plot, bad, dialogue, bad, acting, idioti...",0
49997,I am a Catholic taught in parochial elementary...,i am a catholic taught in parochial elementary...,"[catholic, taught, parochial, elementary, scho...",0
49998,I'm going to have to disagree with the previou...,im going to have to disagree with the previous...,"[im, going, disagree, previous, comment, side,...",0


In [318]:
doc = clean_dataset['lemmatized_tokens']

In [338]:
model_vec = Word2Vec(doc,vector_size = 100, window = 5, min_count = 1, sg = 0) 

In [413]:
def vec_tokens(tokens_lsit) ->list :
    vec_review = []
    for loc in doc:
        len_loc = len(loc)
        vec_loc = np.zeros(100)
        for word in loc:
            predict_model = model_vec.wv[word]
            if len(vec_loc) < 1:
                vec_loc.append(predict_model)
            else:
                vec_loc += predict_model
        vec_review.append(vec_loc / len_loc)

    return vec_review

In [352]:
x = np.array(vec_review)
y = clean_dataset['sentiment']

In [360]:
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state= 42, test_size= 0.2)

In [366]:
log_reg = LogisticRegression(penalty= 'l2')
log_reg.fit(x_train,y_train)
score_train = log_reg.score(x_train,y_train)
score_test = log_reg.score(x_test, y_test)

In [372]:
log_reg_predict = log_reg.predict(x_test)
f1_score_log_reg = f1_score(y_test,log_reg_predict)

In [14]:
X_train, X_test, y_trains, y_tests = train_test_split(clean_dataset['review'],
                                                      clean_dataset['sentiment'],random_state= 42, test_size= 0.2)

In [389]:
test = clean_dataset['review']

In [512]:
vectorizer = CountVectorizer()
x_train_count = vectorizer.fit_transform(X_train)
x_test_count = vectorizer.transform(X_test)

In [513]:
tfidf_transformer = TfidfTransformer()
x_train_tfidf = tfidf_transformer.fit_transform(x_train_count)
x_test_tfidf = tfidf_transformer.transform(x_test_count)

In [519]:
model_three = DecisionTreeClassifier(random_state=42)

parametrs = {
    'max_depth': [20, 30],              # изначально глубина была от 1, 2, 5, 10
    'min_samples_split': [2, 5],
    'min_samples_leaf': [10, 15, 20],   # изначально количество образцов было ot 2, 5, 10
}                                       # измененео для скорости обработки

grid_search = GridSearchCV(estimator=model_three, param_grid=parametrs,
                           cv=2, scoring='accuracy', n_jobs=-1, verbose=1)

grid_search.fit(x_train_tfidf, y_trains)

print(grid_search.best_params_)
print(grid_search.best_score_)

Fitting 2 folds for each of 12 candidates, totalling 24 fits
{'max_depth': 20, 'min_samples_leaf': 15, 'min_samples_split': 2}
0.7248


In [522]:
best_params_three = grid_search.best_estimator_
dtc_predict = best_params_three.predict(x_test_tfidf)

In [526]:
accuracy_dtc = accuracy_score(y_tests, dtc_predict)
f1_score_dtc = f1_score(y_tests, dtc_predict)

In [14]:
def gensim_tokens(data) -> list:
    tokens_list = []
    
    for tokens in data:
        sec_tokens = simple_preprocess(tokens, deacc=True, min_len=3, max_len=30)
        tokens_list.append(sec_tokens)

    return tokens_list

x_train_tokens_gensim = gensim_tokens(X_train)
x_test_tokens_gensim = gensim_tokens(X_test)

In [16]:
def stopworrds_remove(data) ->list:
    remove_list = []

    for loc in data:
        sec_remove = []
        for word in loc:
            if word not in STOPWORDS:
                sec_remove.append(word)
        remove_list.append(sec_remove)
    return remove_list

x_train_clean_tokens = stopworrds_remove(x_train_tokens_gensim)
x_test_clean_tokens = stopworrds_remove(x_test_tokens_gensim)

In [17]:
diction = Dictionary(x_train_clean_tokens)

In [18]:
bow_corpus_train = [diction.doc2bow(doc) for doc in x_train_clean_tokens]
bow_corpus_test = [diction.doc2bow(doc) for doc in x_test_clean_tokens]

In [19]:
model_train = TfidfModel(bow_corpus_train)

In [20]:
tfidf_corpus_train = model_train[bow_corpus_train]
tfidf_corpus_test = model_train[bow_corpus_test]

In [21]:
X_train_csc = matutils.corpus2csc(tfidf_corpus_train)  
X_test_csc = matutils.corpus2csc(tfidf_corpus_test)

In [22]:
X_train_gens = X_train_csc.T.tocsr()
X_test_gens  = X_test_csc.T.tocsr()

In [26]:
catbc = CatBoostClassifier(iterations=50,
    learning_rate=0.1,
    depth=2,
    loss_function='Logloss',
    verbose=False)
catbc.fit(X_train_gens, y_trains)

CatBoostClassifier(depth=2, iterations=50, learning_rate=0.1, loss_function='Logloss', verbose=False)

In [27]:
catbc_predict = catbc.predict(X_test_gens)

In [28]:
catbc_accuracy = accuracy_score(y_tests, catbc_predict)
catbc_f1_score = f1_score(y_tests, catbc_predict)

In [30]:
catbc_f1_score

0.7923230128553322

In [12]:
nlp = spacy.load("en_core_web_sm")

In [14]:
text = clean_dataset['review']

In [ ]:
docs = list(nlp.pipe(text, disable=["parser", "ner"]))

In [ ]:
qwe1 = []
for doc in text:
    sec = nlp(doc)
    clean = [tok.text for tok in sec if not tok.is_stop and not tok.is_punct and tok.is_alpha]
    qwe1.append(clean)